# 05 - Kernel Fitting

**Purpose:** Model the *shape* of how economic shocks propagate to food insecurity.

**Key Question:** Is the response immediate (delta), gradual decay (exponential), or delayed-peak (gamma)?

## Transfer Function Model

We model food insecurity as a convolution of energy price shocks with a response kernel:

$$D(t) = (S * h)(t) + \epsilon(t)$$

Where:
- $D(t)$ = Food insecurity at time $t$
- $S(t)$ = Energy price shock signal
- $h(\tau)$ = **Systemic Vulnerability Kernel** (what we're fitting)
- $\epsilon(t)$ = Noise

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import signal
from scipy.optimize import minimize
from scipy.stats import gamma as gamma_dist
from scipy.special import factorial
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Data

In [ ]:
aligned = pl.read_parquet(PROCESSED_DATA / 'aligned_timeseries.parquet').to_pandas()
aligned['date'] = pd.to_datetime(aligned['date'])

# Extract signals
energy = aligned['energy_price'].values
food = aligned['food_insecurity'].values
dates = aligned['date'].values

# Compute the "shock" signal as the change in energy price
# This represents the delta-function impulses we're convolving with the kernel
energy_shock = np.diff(energy, prepend=energy[0])

print(f"Time series length: {len(energy)} months")
print(f"Energy price range: £{energy.min():.0f} - £{energy.max():.0f}")
print(f"Food insecurity range: {food.min():.1f}% - {food.max():.1f}%")

## 2. Define Candidate Kernels

We test three kernel shapes representing different hypotheses about how shocks propagate:

| Kernel | Shape | Interpretation |
|--------|-------|----------------|
| **Delta** | Spike at lag=0 | Instantaneous pass-through |
| **Exponential** | Decay from t=0 | Immediate impact, fading over time |
| **Gamma** | Delayed peak | Shock takes time to manifest, then fades |

In [ ]:
def kernel_delta(t, amplitude=1.0):
    """
    Delta kernel: All impact at t=0.
    
    Represents instantaneous pass-through of shock to response.
    """
    k = np.zeros_like(t, dtype=float)
    k[0] = amplitude
    return k


def kernel_exponential(t, amplitude=1.0, decay_rate=0.3):
    """
    Exponential decay kernel: h(t) = A * exp(-λt)
    
    Represents immediate impact that decays over time.
    decay_rate (λ) controls how fast the effect diminishes.
    """
    k = amplitude * np.exp(-decay_rate * t)
    k[t < 0] = 0
    return k / k.sum()  # Normalize


def kernel_gamma(t, amplitude=1.0, shape=2.0, scale=1.5):
    """
    Gamma kernel: h(t) = A * Gamma(t; k, θ)
    
    Represents delayed peak response.
    - shape (k): Controls how delayed the peak is
    - scale (θ): Controls the spread
    - Peak occurs at t = (shape-1) * scale
    """
    k = amplitude * gamma_dist.pdf(t, a=shape, scale=scale)
    k[t < 0] = 0
    if k.sum() > 0:
        k = k / k.sum()  # Normalize
    return k


# Visualize kernel shapes
t = np.arange(0, 12)  # 12 months

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(t, kernel_delta(t), color='#264653', alpha=0.7)
axes[0].set_title('Delta Kernel\n(Instantaneous)', fontsize=12)
axes[0].set_xlabel('Lag (months)')
axes[0].set_ylabel('Weight')

axes[1].bar(t, kernel_exponential(t, decay_rate=0.5), color='#2A9D8F', alpha=0.7)
axes[1].set_title('Exponential Kernel\n(Immediate + Decay)', fontsize=12)
axes[1].set_xlabel('Lag (months)')

axes[2].bar(t, kernel_gamma(t, shape=3, scale=1.0), color='#E76F51', alpha=0.7)
axes[2].set_title('Gamma Kernel\n(Delayed Peak)', fontsize=12)
axes[2].set_xlabel('Lag (months)')

plt.tight_layout()
plt.savefig(OUTPUTS / 'kernel_shapes.png', dpi=150)
plt.show()

## 3. Fit Kernels to Data

For each kernel type, we find the parameters that minimize the prediction error.

In [ ]:
def predict_with_kernel(shock_signal, kernel, baseline=0):
    """
    Convolve shock signal with kernel to predict response.
    """
    # Convolve and take the 'same' length output
    convolved = np.convolve(shock_signal, kernel, mode='full')[:len(shock_signal)]
    return baseline + convolved


def fit_kernel(shock_signal, response, kernel_func, param_bounds, n_kernel_points=12):
    """
    Fit kernel parameters by minimizing MSE between predicted and actual response.
    
    Returns:
        best_params: Optimal parameters
        best_kernel: Kernel at optimal parameters
        r_squared: Coefficient of determination
        predicted: Predicted response
    """
    t = np.arange(n_kernel_points)
    
    def objective(params):
        kernel = kernel_func(t, *params)
        # Add baseline and scale as fitted parameters
        baseline = params[-2] if len(params) > 2 else response.mean()
        scale = params[-1] if len(params) > 2 else 1.0
        
        predicted = baseline + scale * np.convolve(shock_signal, kernel, mode='full')[:len(shock_signal)]
        mse = np.mean((response - predicted) ** 2)
        return mse
    
    # Try multiple starting points
    best_result = None
    best_mse = np.inf
    
    for _ in range(10):
        x0 = [np.random.uniform(b[0], b[1]) for b in param_bounds]
        result = minimize(objective, x0, bounds=param_bounds, method='L-BFGS-B')
        if result.fun < best_mse:
            best_mse = result.fun
            best_result = result
    
    # Extract results
    best_params = best_result.x
    kernel = kernel_func(t, *best_params[:-2]) if len(best_params) > 2 else kernel_func(t, *best_params)
    
    baseline = best_params[-2] if len(best_params) > 2 else response.mean()
    scale = best_params[-1] if len(best_params) > 2 else 1.0
    predicted = baseline + scale * np.convolve(shock_signal, kernel, mode='full')[:len(shock_signal)]
    
    # R-squared
    ss_res = np.sum((response - predicted) ** 2)
    ss_tot = np.sum((response - response.mean()) ** 2)
    r_squared = 1 - (ss_res / ss_tot)
    
    return best_params, kernel, r_squared, predicted

In [ ]:
# Standardize signals for fitting
energy_norm = (energy - energy.mean()) / energy.std()
food_norm = (food - food.mean()) / food.std()

# Use level (not shock) as predictor - simpler model
shock = energy_norm
response = food_norm

t = np.arange(12)

results = {}

In [ ]:
# Fit Delta kernel (just find optimal lag)
print("Fitting Delta kernel...")

best_r2_delta = -np.inf
best_lag_delta = 0

for lag in range(12):
    kernel = np.zeros(12)
    kernel[lag] = 1.0
    
    predicted = np.convolve(shock, kernel, mode='full')[:len(shock)]
    
    # Fit linear regression for scale and baseline
    X = np.column_stack([np.ones_like(predicted), predicted])
    coeffs = np.linalg.lstsq(X, response, rcond=None)[0]
    predicted_scaled = X @ coeffs
    
    ss_res = np.sum((response - predicted_scaled) ** 2)
    ss_tot = np.sum((response - response.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    if r2 > best_r2_delta:
        best_r2_delta = r2
        best_lag_delta = lag
        best_predicted_delta = predicted_scaled
        best_kernel_delta = kernel

results['delta'] = {
    'r_squared': best_r2_delta,
    'params': {'lag': best_lag_delta},
    'kernel': best_kernel_delta,
    'predicted': best_predicted_delta
}

print(f"  Best lag: {best_lag_delta} months")
print(f"  R-squared: {best_r2_delta:.3f}")

In [ ]:
# Fit Exponential kernel
print("\nFitting Exponential kernel...")

best_r2_exp = -np.inf
best_decay_exp = 0.5

for decay in np.linspace(0.1, 2.0, 50):
    kernel = kernel_exponential(t, decay_rate=decay)
    
    predicted = np.convolve(shock, kernel, mode='full')[:len(shock)]
    
    X = np.column_stack([np.ones_like(predicted), predicted])
    coeffs = np.linalg.lstsq(X, response, rcond=None)[0]
    predicted_scaled = X @ coeffs
    
    ss_res = np.sum((response - predicted_scaled) ** 2)
    ss_tot = np.sum((response - response.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    if r2 > best_r2_exp:
        best_r2_exp = r2
        best_decay_exp = decay
        best_predicted_exp = predicted_scaled
        best_kernel_exp = kernel

results['exponential'] = {
    'r_squared': best_r2_exp,
    'params': {'decay_rate': best_decay_exp},
    'kernel': best_kernel_exp,
    'predicted': best_predicted_exp
}

print(f"  Best decay rate: {best_decay_exp:.2f}")
print(f"  R-squared: {best_r2_exp:.3f}")

In [ ]:
# Fit Gamma kernel
print("\nFitting Gamma kernel...")

best_r2_gamma = -np.inf
best_shape_gamma = 2.0
best_scale_gamma = 1.0

for shape in np.linspace(1.5, 6.0, 20):
    for scale in np.linspace(0.5, 3.0, 20):
        kernel = kernel_gamma(t, shape=shape, scale=scale)
        
        if kernel.sum() == 0:
            continue
            
        predicted = np.convolve(shock, kernel, mode='full')[:len(shock)]
        
        X = np.column_stack([np.ones_like(predicted), predicted])
        coeffs = np.linalg.lstsq(X, response, rcond=None)[0]
        predicted_scaled = X @ coeffs
        
        ss_res = np.sum((response - predicted_scaled) ** 2)
        ss_tot = np.sum((response - response.mean()) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        
        if r2 > best_r2_gamma:
            best_r2_gamma = r2
            best_shape_gamma = shape
            best_scale_gamma = scale
            best_predicted_gamma = predicted_scaled
            best_kernel_gamma = kernel

peak_month = (best_shape_gamma - 1) * best_scale_gamma

results['gamma'] = {
    'r_squared': best_r2_gamma,
    'params': {'shape': best_shape_gamma, 'scale': best_scale_gamma, 'peak_month': peak_month},
    'kernel': best_kernel_gamma,
    'predicted': best_predicted_gamma
}

print(f"  Best shape (k): {best_shape_gamma:.2f}")
print(f"  Best scale (θ): {best_scale_gamma:.2f}")
print(f"  Peak response at: {peak_month:.1f} months")
print(f"  R-squared: {best_r2_gamma:.3f}")

## 4. Compare Kernel Performance

In [ ]:
print("\n" + "="*60)
print("KERNEL COMPARISON")
print("="*60)
print(f"\n{'Kernel':<15} {'R-squared':<12} {'Key Parameter':<25}")
print("-"*52)
print(f"{'Delta':<15} {results['delta']['r_squared']:<12.3f} lag = {results['delta']['params']['lag']} months")
print(f"{'Exponential':<15} {results['exponential']['r_squared']:<12.3f} decay = {results['exponential']['params']['decay_rate']:.2f}")
print(f"{'Gamma':<15} {results['gamma']['r_squared']:<12.3f} peak = {results['gamma']['params']['peak_month']:.1f} months")

# Identify best
best_kernel = max(results.keys(), key=lambda k: results[k]['r_squared'])
print(f"\nBest fit: {best_kernel.upper()} kernel (R² = {results[best_kernel]['r_squared']:.3f})")

In [ ]:
# Visualize fitted kernels
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = {'delta': '#264653', 'exponential': '#2A9D8F', 'gamma': '#E76F51'}

for ax, (name, res) in zip(axes, results.items()):
    ax.bar(t, res['kernel'], color=colors[name], alpha=0.7, edgecolor='black')
    ax.set_title(f"{name.capitalize()} Kernel\nR² = {res['r_squared']:.3f}", fontsize=12)
    ax.set_xlabel('Lag (months)')
    ax.set_ylabel('Weight')
    ax.set_xticks(range(0, 12, 2))

# Highlight best
best_idx = list(results.keys()).index(best_kernel)
axes[best_idx].set_title(f"{best_kernel.capitalize()} Kernel\nR² = {results[best_kernel]['r_squared']:.3f} ★ BEST", 
                          fontsize=12, color='red')

plt.tight_layout()
plt.savefig(OUTPUTS / 'fitted_kernels.png', dpi=150)
plt.show()

In [ ]:
# Visualize predictions vs actual
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(dates, response, 'ko-', linewidth=2, markersize=5, label='Actual Food Insecurity')

for name, res in results.items():
    linestyle = '-' if name == best_kernel else '--'
    alpha = 1.0 if name == best_kernel else 0.6
    ax.plot(dates, res['predicted'], linestyle=linestyle, color=colors[name], 
            linewidth=2, alpha=alpha, label=f"{name.capitalize()} (R²={res['r_squared']:.3f})")

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Food Insecurity (standardized)', fontsize=12)
ax.set_title('Kernel Model Predictions vs Actual Food Insecurity', fontsize=14)
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig(OUTPUTS / 'kernel_predictions.png', dpi=150)
plt.show()

## 5. Interpretation

In [ ]:
print("\n" + "="*60)
print("KERNEL FITTING - INTERPRETATION")
print("="*60)

if best_kernel == 'delta':
    print(f"""
BEST FIT: DELTA KERNEL

The delta kernel suggests that energy price changes have an
essentially instantaneous effect on food insecurity, with a
fixed delay of {results['delta']['params']['lag']} months.

Policy Implication:
  - Response is immediate but lagged
  - Intervention window is fixed at ~{results['delta']['params']['lag']} months
  - No lingering effects after the initial response
""")

elif best_kernel == 'exponential':
    half_life = np.log(2) / results['exponential']['params']['decay_rate']
    print(f"""
BEST FIT: EXPONENTIAL KERNEL

The exponential kernel suggests that energy price changes have
an immediate impact that decays over time.

Parameters:
  - Decay rate: {results['exponential']['params']['decay_rate']:.2f}
  - Half-life: {half_life:.1f} months

Policy Implication:
  - Impact is felt immediately
  - Effect diminishes by half every {half_life:.1f} months
  - Intervention most effective at shock onset
""")

elif best_kernel == 'gamma':
    print(f"""
BEST FIT: GAMMA KERNEL

The gamma kernel suggests that energy price changes take time
to fully manifest, with a delayed peak response.

Parameters:
  - Shape (k): {results['gamma']['params']['shape']:.2f}
  - Scale (θ): {results['gamma']['params']['scale']:.2f}
  - Peak response: {results['gamma']['params']['peak_month']:.1f} months after shock

Policy Implication:
  - Impact builds gradually after a shock
  - Peak vulnerability occurs at ~{results['gamma']['params']['peak_month']:.0f} months
  - Intervention window extends from 0 to ~{results['gamma']['params']['peak_month']*2:.0f} months
  - This matches a model where households:
    1. Initially absorb shock using savings
    2. Gradually deplete buffers
    3. Peak food insecurity when buffers exhausted
""")

## 6. Save Results

In [ ]:
import json

# Prepare results for JSON (convert numpy types)
json_results = {}
for name, res in results.items():
    json_results[name] = {
        'r_squared': float(res['r_squared']),
        'params': {k: float(v) for k, v in res['params'].items()},
        'kernel': [float(x) for x in res['kernel']]
    }

json_results['best_kernel'] = best_kernel

with open(OUTPUTS / 'kernel_fitting_results.json', 'w') as f:
    json.dump(json_results, f, indent=2)

print("Saved:")
print(f"  - {OUTPUTS / 'kernel_fitting_results.json'}")
print(f"  - {OUTPUTS / 'kernel_shapes.png'}")
print(f"  - {OUTPUTS / 'fitted_kernels.png'}")
print(f"  - {OUTPUTS / 'kernel_predictions.png'}")